<a href="https://colab.research.google.com/github/kandinz/Omni-TTS/blob/main/colab_batch_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiên Đoàn TTS — Tạo Audio Hàng Loạt từ JSON (Batch Audio Generator)

> **GPU:** T4 16GB (tự động) · **Chế độ:** Batch Generation từ file JSON · **Tốc độ:** Cài đặt ~2 phút (1 lần đầu), sinh audio ~3-5s/cảnh

## 🚀 Hướng dẫn sử dụng

1. **Chỉnh sửa cấu hình** ở **Block Cấu hình (Cell bên dưới)** nếu muốn đổi danh sách JSON, tốc độ, ngôn ngữ...
2. **Bấm Runtime → Run all (hoặc nhấn Ctrl + F9)** để notebook tự động thực thi từ đầu đến cuối.

In [1]:
# ==============================================================================
# ⚙️ BẢNG CẤU HÌNH TÙY CHỈNH HỆ THỐNG (CENTRAL CONFIGURATION)
# Tất cả cài đặt nằm tại đây. Chỉnh sửa xong chỉ cần nhấn Ctrl + F9 để chạy!
# ==============================================================================

# 1. Cấu hình Giọng đọc Mẫu (Voice Reference)
VOICE_SAMPLE_URL = "https://raw.githubusercontent.com/kandinz/Omni-TTS/refs/heads/main/voice%20sample/Gum%202.m4a"
VOICE_SAMPLE_FILE = "voice_sample.mp3"

# 2. Cấu hình Kịch bản Đầu vào (Input JSON)
JSON_DATA_URL = "https://raw.githubusercontent.com/kandinz/Omni-TTS/refs/heads/main/voiceover-scripts/script1.json"                 # Tải kịch bản từ URL (raw GitHub, JSONBin...). Để trống nếu dùng file local
JSON_FILE_PATH = "voiceover_scenes.json"  # Đọc từ file này nếu không lấy được từ URL

# Kịch bản mặc định (sử dụng khi không lấy được từ URL và File local)
DEFAULT_JSON_DATA = [
  {
    "filename": "01_hook.wav",
    "text": "Mỗi ngày bạn mất 3-4 tiếng cho những việc lặp đi lặp lại? Đã đến lúc để AI Agent làm thay bạn!"
  },
  {
    "filename": "02_concept.wav",
    "text": "Khác với Chatbot chỉ biết trả lời, AI Agent là một trợ lý thông minh có khả năng tự lập kế hoạch và chủ động thực thi công việc từ A đến Z."
  },
  {
    "filename": "03_pdf.wav",
    "text": "Ví dụ 1: Bạn có 50 hóa đơn PDF? Agent tự đọc, bóc tách dữ liệu và điền chuẩn xác vào file Excel chỉ trong vài giây."
  },
  {
    "filename": "04_email.wav",
    "text": "Ví dụ 2: Bạn cần báo cáo tuần? Agent tự động tổng hợp số liệu CRM và gửi email báo cáo đúng 5 giờ chiều."
  },
  {
    "filename": "05_research.wav",
    "text": "Ví dụ 3: Cần đọc tài liệu 100 trang? Agent tóm tắt thành 3 ý chính trong vài giây."
  },
  {
    "filename": "06_outro.wav",
    "text": "Tập trung vào giá trị thật, để AI Agent lo phần còn lại! Trải nghiệm AI Agent ngay hôm nay!"
  }
]

# 3. Cấu hình Tham số Sinh Giọng Nói (TTS Settings)
LANGUAGE = "vi"                    # Ngôn ngữ giọng đọc: 'vi' (Tiếng Việt)
VOICE_SPEED = 0.95                  # Tốc độ đọc (Mặc định: 0.95)
SILENCE_BETWEEN_PARAGRAPHS = 0.3    # Khoảng lặng giữa các đoạn (giây)

# 4. Cấu hình OmniVoice Advanced Config
NUM_STEPS = 32                      # Số bước suy luận (32 = nhanh & chất lượng)
GUIDANCE_SCALE = 1.8               # Độ bám giọng đọc mẫu (1.5 - 2.0)
DENOISE = True                     # Khử nhiễu audio kết quả
PREPROCESS_PROMPT = True           # Tiền xử lý prompt giọng đọc mẫu
POSTPROCESS_OUTPUT = True          # Hậu xử lý âm thanh đầu ra
POSITION_TEMP = 5.0                # Nhiệt độ vị trí
CLASS_TEMP = 0.2                   # Nhiệt độ phân loại
PAD_DURATION = 0.1                 # Thêm khoảng đệm đầu/cuối (s)
FADE_DURATION = 0.1                # Thời gian fade in/out (s)

# 5. Cấu hình Đầu ra & Đóng gói (Output Settings)
OUTPUT_DIR = "output_audio"
ZIP_FILENAME = "output_audio.zip"
AUTO_DOWNLOAD_ZIP = True           # Tự động tải file ZIP về trình duyệt sau khi chạy xong

print('✅ Đã nạp bảng cấu hình tùy chỉnh thành công!')

✅ Đã nạp bảng cấu hình tùy chỉnh thành công!


In [2]:
# === BLOCK 1: CÀI ĐẶT THƯ VIỆN & TẢI VOICE SAMPLE ===
import os, sys

print('🚀 [1/5] Đang kiểm tra môi trường cài đặt...')

try:
    import omnivoice
    print('✅ OmniVoice đã được cài đặt sẵn!')
except ImportError:
    print('📦 Đang cài đặt thư viện cần thiết (~2 phút)...')
    !pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4" scipy tqdm
    !pip uninstall -y transformers
    !pip install -q "transformers>=5.3.0"
    print('✅ Cài đặt thư viện hoàn tất!')

# Tải file giọng mẫu dựa theo VOICE_SAMPLE_FILE & VOICE_SAMPLE_URL trong config
if not os.path.exists(VOICE_SAMPLE_FILE):
    print(f'📥 Đang tải voice sample ({VOICE_SAMPLE_FILE})...')
    !wget -q "{VOICE_SAMPLE_URL}" -O "{VOICE_SAMPLE_FILE}"
    print(f'✅ Đã tải thành công {VOICE_SAMPLE_FILE}!')
else:
    print(f'✅ File {VOICE_SAMPLE_FILE} đã sẵn sàng!')

🚀 [1/5] Đang kiểm tra môi trường cài đặt...
📦 Đang cài đặt thư viện cần thiết (~2 phút)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.1 MB/s eta 0:00:00
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.5 MB/s eta 0:00:00
✅ Cài đặt thư viện hoàn tất!
📥 Đang tải voice sample (voice_sample.mp3)...
✅ Đã tải thành công voice_sample.mp3!


In [3]:
# === BLOCK 2: KHỞI TẠO OMNIVOICE MODEL & VOICE PROMPT ===
print('🤖 [2/5] Đang khởi động OmniVoice Model...')

import logging, time, json, re
import numpy as np
import torch
from scipy.io import wavfile
from tqdm.notebook import tqdm

# Shim: AutoFeatureExtractor removed in transformers 5.x
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Kiểm tra GPU CUDA
if torch.cuda.is_available():
    print(f'⚡ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ CẢNH BÁO: Không tìm thấy GPU! Hãy chọn Runtime > Change runtime type > T4 GPU.')

# Nạp Model 1 lần duy nhất vào globals
if 'model' not in globals():
    DEVICE = get_best_device()
    print(f'🧠 Đang nạp model OmniVoice vào {DEVICE} (float16)...')
    model = OmniVoice.from_pretrained(
        'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
    )
    SAMPLING_RATE = model.sampling_rate
    print(f'✅ Model ready — Sampling Rate: {SAMPLING_RATE}Hz')

    print(f'🎙️ Đang khởi tạo Voice Clone Prompt từ {VOICE_SAMPLE_FILE}...')
    VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio=VOICE_SAMPLE_FILE)
    print('✅ Voice Prompt ready!')
else:
    print('✅ Model và Voice Prompt đã được load sẵn từ trước!')

🤖 [2/5] Đang khởi động OmniVoice Model...


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


⚡ GPU: Tesla T4
🧠 Đang nạp model OmniVoice vào cuda (float16)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

...

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

✅ Model ready — Sampling Rate: 24000Hz
🎙️ Đang khởi tạo Voice Clone Prompt từ voice_sample.mp3...


/usr/local/lib/python3.12/dist-packages/omnivoice/utils/audio.py:63: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(audio_path, sr=None, mono=False)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


EOFError: 

In [ ]:
# === BLOCK 3: NẠP KỊCH BẢN JSON ===
print('📄 [3/5] Nạp dữ liệu kịch bản JSON...')

import requests

scenes = None

# 1. Ưu tiên thử nạp kịch bản từ URL (JSON_DATA_URL)
if 'JSON_DATA_URL' in globals() and JSON_DATA_URL and JSON_DATA_URL.strip():
    url = JSON_DATA_URL.strip()
    print(f"🌐 Đang nạp kịch bản từ URL: {url}...")
    try:
        res = requests.get(url, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if data:
                scenes = data
                print("✅ Nạp kịch bản từ URL thành công!")
            else:
                print("⚠️ Dữ liệu JSON từ URL rỗng.")
        else:
            print(f"⚠️ Không thể tải JSON từ URL (Mã HTTP: {res.status_code}).")
    except Exception as e:
        print(f"⚠️ Lỗi khi tải kịch bản từ URL: {e}")

# 2. Nếu từ link không có data, mới nạp từ file JSON_FILE_PATH
if not scenes:
    if os.path.exists(JSON_FILE_PATH):
        print(f"📂 Phát hiện file '{JSON_FILE_PATH}', đang nạp kịch bản từ file...")
        try:
            with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if data:
                    scenes = data
                    print(f"✅ Nạp kịch bản từ file '{JSON_FILE_PATH}' thành công!")
                else:
                    print(f"⚠️ File '{JSON_FILE_PATH}' chứa dữ liệu rỗng.")
        except Exception as e:
            print(f"⚠️ Lỗi khi đọc file '{JSON_FILE_PATH}': {e}")
    else:
        print(f"ℹ️ Chưa upload hoặc không tìm thấy file '{JSON_FILE_PATH}'.")

# 3. Nếu vẫn không có data thì dùng DEFAULT_JSON_DATA
if not scenes:
    print("ℹ️ Chưa nạp được kịch bản từ URL hoặc File local — Sử dụng danh sách kịch bản DEFAULT_JSON_DATA từ Block Config...")
    scenes = DEFAULT_JSON_DATA

print(f"🎬 Tổng số cảnh cần tạo audio: {len(scenes)}")
for idx, item in enumerate(scenes, 1):
    print(f"   [{idx}] File: {item['filename']} | Nội dung: \"{item['text'][:45]}...\"")


In [ ]:
# === BLOCK 4: SINH AUDIO HÀNG LOẠT (OPTIMIZED BATCH GENERATION) ===
print('⚡ [4/5] Đang tiến hành tạo audio hàng loạt...')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Khởi tạo cấu hình từ Bảng Config
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=NUM_STEPS, guidance_scale=GUIDANCE_SCALE,
    denoise=DENOISE, preprocess_prompt=PREPROCESS_PROMPT, postprocess_output=POSTPROCESS_OUTPUT,
    position_temperature=POSITION_TEMP, class_temperature=CLASS_TEMP,
    pad_duration=PAD_DURATION, fade_duration=FADE_DURATION,
)

def generate_scene_waveform(text: str):
    text = text.strip()
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        return None
    if len(paragraphs) == 1:
        audio = model.generate(
            text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
            language=LANGUAGE, speed=VOICE_SPEED, generation_config=GEN_CFG
        )[0]
    else:
        audios = []
        for i, p in enumerate(paragraphs):
            a = model.generate(
                text=p, voice_clone_prompt=VOICE_PROMPT,
                language=LANGUAGE, speed=VOICE_SPEED, generation_config=GEN_CFG
            )[0]
            audios.append(a)
            if i < len(paragraphs) - 1:
                audios.append(np.zeros(int(SAMPLING_RATE * SILENCE_BETWEEN_PARAGRAPHS)))
        audio = np.concatenate(audios)
    waveform = (audio * 32767).astype(np.int16)
    return waveform

start_time = time.time()
generated_count = 0

with torch.inference_mode():
    for item in tqdm(scenes, desc="Đang sinh Audio các cảnh"):
        filename = item['filename']
        text = item['text']
        save_path = os.path.join(OUTPUT_DIR, filename)

        waveform = generate_scene_waveform(text)
        if waveform is not None:
            wavfile.write(save_path, SAMPLING_RATE, waveform)
            generated_count += 1

elapsed = time.time() - start_time
print(f"✅ ĐÃ HOÀN THÀNH: Sinh xong {generated_count}/{len(scenes)} file WAV trong {elapsed:.1f} giây (Trung bình {elapsed/max(1, generated_count):.1f}s/cảnh)!")

In [ ]:
# === BLOCK 5: ĐÓNG GÓI ZIP, TẢI VỀ & NGHE THỬ TRỰC TIẾP ===
print('📦 [5/5] Đóng gói ZIP và kích hoạt tải về...')

import shutil
from IPython.display import Audio, display, HTML

# 1. Đóng gói thư mục thành file zip
base_zip_name = os.path.splitext(ZIP_FILENAME)[0]
shutil.make_archive(base_zip_name, "zip", OUTPUT_DIR)
print(f"✅ Đã nén thành công: {ZIP_FILENAME}")

# 2. Tự động kích hoạt trình download trên browser (Google Colab)
if AUTO_DOWNLOAD_ZIP:
    try:
        from google.colab import files
        print(f"⬇️ Đang tải xuống file {ZIP_FILENAME} tự động...")
        files.download(ZIP_FILENAME)
    except ImportError:
        print(f"ℹ️ File ZIP đã lưu tại địa chỉ local: {os.path.abspath(ZIP_FILENAME)}")

# 3. Trình nghe thử kết quả audio trực tiếp ngay trên Notebook
display(HTML("<h3>🎧 Trình Nghe Thử Các Cảnh Video Audio:</h3>"))
for item in scenes:
    fname = item['filename']
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        display(HTML(f"<p style='margin-bottom:2px;'>🔊 <b>{fname}</b> — <i>\"{item['text']}\"</i></p>"))
        display(Audio(fpath))
